# Projeto 1 - C11: Ciência de Dados
**Instituto Nacional de Telecomunicações (Inatel)**  
**Docente:** Prof. Dr. Renzo Mesquita Paranaíba  

### 👥 Integrantes do Grupo:
- **Igor Araujo** (479 - GES)
- **Kevin Lopes do Vale** (1955 - GEC)
- **Felipe Fróes Pereira** (566 - GES)

---

## 📌 Contexto do Estudo
Este projeto investiga o impacto da Inteligência Artificial Generativa sobre a rotina, produtividade e tempo economizado de estudantes e profissionais com base no dataset real do Kaggle: **AI Usage and Impact on Students & Professionals**.

Neste notebook:
- **Etapa 1:** Importação de pacotes e higienização/pré-processamento da base de dados.
- **Etapa 2 (Nossa parte):** Resolução detalhada das **Perguntas 1 a 5** com estatísticas descritivas, gráficos comparativos e conclusões analíticas.
- **Etapa 3 (Área do Colega):** Estrutura comentada e pronta para a implementação das **Perguntas 6 a 10**.


---
## 1. Importação das Bibliotecas e Configuração do Ambiente


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configuração visual profissional para todos os gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("✔ Bibliotecas importadas com sucesso!")


---
## 2. Carregamento e Pré-Processamento dos Dados

Para assegurar a qualidade dos dados e insights estatisticamente confiáveis, aplicamos:
1. **Deduplicação:** Remoção de registros duplicados no dataset original.
2. **Conversão de Tipos:** Extração numérica de strings (ex: `'2 hrs'`, `'$0 '`, `'$1,884'`).
3. **Padronização Categórica:** Unificação de grafias para ferramentas (`'ChatGPT'`, `'Chat GPT'`), gêneros (`'Male'`, `'M'`), países e níveis de escolaridade.
4. **Engenharia de Atributos:** Criação da métrica de eficiência `Time_Saved_Per_Usage_Hour` e segmentação `Paid_User`.


In [ ]:
# Carregamento da função modular desenvolvida em src/analises.py
sys.path.append(os.path.abspath('..'))
from src.analises import carregar_e_limpar_dados

df = carregar_e_limpar_dados('../data/AI_Usage_and_Impact_on_Students_and_Professionals.csv')

print(f"Dimensões do dataset limpo: {df.shape[0]} linhas x {df.shape[1]} colunas\n")
display(df.head())


In [ ]:
# Verificação das variáveis e tipos tratados
df.info()


---
# 📊 Resposta às Perguntas de Pesquisa (1 a 5)

---
### Pergunta 1: Estudantes vs. Profissionais
> **Qual dos dois perfis despende mais horas diárias utilizando ferramentas de IA (`AI_Usage_Hours_Per_Day`) e quem reporta maior índice de produtividade (`Productivity_Score`)?**

* **Métricas Analisadas:** Horas de uso diário de IA e pontuação de produtividade (0 a 10).
* **Gráfico Ideal:** Boxplot Comparativo Lado a Lado (avalia medianas, quartis e outliers sem ocultar dispersão).


In [ ]:
# Estatísticas descritivas para a Pergunta 1
resumo_p1 = df.groupby('User_Type')[['AI_Usage_Hours_Per_Day', 'Productivity_Score']].agg(
    ['count', 'mean', 'median', 'std']
).round(2)

display(resumo_p1)


In [ ]:
# Visualização gráfica - Pergunta 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Horas diárias de uso
sns.boxplot(
    data=df, x='User_Type', y='AI_Usage_Hours_Per_Day',
    hue='User_Type', legend=False, palette=['#3470a3', '#e27c38'], ax=axes[0]
)
axes[0].set_title('Horas Diárias de Uso de IA por Perfil', fontweight='bold')
axes[0].set_xlabel('Perfil do Usuário')
axes[0].set_ylabel('Uso Diário (horas)')

# 2. Produtividade reportada
sns.boxplot(
    data=df, x='User_Type', y='Productivity_Score',
    hue='User_Type', legend=False, palette=['#3470a3', '#e27c38'], ax=axes[1]
)
axes[1].set_title('Produtividade Reportada por Perfil', fontweight='bold')
axes[1].set_xlabel('Perfil do Usuário')
axes[1].set_ylabel('Productivity Score (0 a 10)')

plt.tight_layout()
plt.show()


**💡 Insights e Conclusões (Pergunta 1):**
- **Tempo de Uso:** Estudantes e profissionais apresentam mediana de uso bastante próxima (~2.0 a 2.1 horas diárias). No entanto, estudantes possuem ligeiramente maior média global (~3.01h vs ~2.87h), com alguns outliers ultrapassando 8 horas diárias de uso.
- **Produtividade Percebida:** Profissionais reportam maior produtividade percebida (média de **5.28** e mediana de **5.10**) em relação aos estudantes (média de **5.19** e mediana de **4.90**). Isso sugere que profissionais conseguem converter as ferramentas em impacto corporativo imediato com fluxos de trabalho mais definidos.


---
### Pergunta 2: Preferência de Ferramentas de IA
> **Quais são as ferramentas de IA mais populares no dataset e como essa distribuição se divide entre estudantes e profissionais?**

* **Métricas Analisadas:** Frequência absoluta e relativa das soluções de IA adotadas por perfil.
* **Gráfico Ideal:** Gráfico de Barras Horizontais Agrupadas por Perfil.


In [ ]:
# Tabela cruzada de ferramentas por perfil de usuário
tabela_cruzada = pd.crosstab(df['AI_Tool'], df['User_Type'], margins=True).sort_values('All', ascending=False)
display(tabela_cruzada)


In [ ]:
# Visualização gráfica - Pergunta 2
df_p2 = df.dropna(subset=['AI_Tool', 'User_Type']).groupby(['AI_Tool', 'User_Type']).size().reset_index(name='Quantidade')
ordem_ferramentas = df['AI_Tool'].value_counts().sort_values(ascending=True).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_p2, y='AI_Tool', x='Quantidade', hue='User_Type',
    order=ordem_ferramentas, palette=['#3470a3', '#e27c38'], ax=ax
)

ax.set_title('Preferência de Ferramentas de IA: Estudantes vs. Profissionais', fontweight='bold')
ax.set_xlabel('Número de Usuários')
ax.set_ylabel('Ferramenta de IA')
ax.legend(title='Perfil de Usuário')

plt.tight_layout()
plt.show()


**💡 Insights e Conclusões (Pergunta 2):**
- **Liderança Absoluta do ChatGPT:** O ChatGPT é a ferramenta predominante de longe em ambos os perfis (119 usuários no total, sendo 62 estudantes e 57 profissionais).
- **Competição de Segundo Escalão:** **Google Gemini** (39 usuários) e **Microsoft Copilot** (33 usuários) formam a segunda força mais expressiva. O Copilot e o Claude têm forte aderência entre estudantes universitários, enquanto o **Perplexity AI** mostra maior tração relativa entre profissionais que realizam pesquisas aprofundadas.


---
### Pergunta 3: Tempo de Uso vs. Economia Real
> **Existe uma correlação positiva direta entre as horas diárias gastas com IA (`AI_Usage_Hours_Per_Day`) e o tempo semanal economizado (`Time_Saved_Hours_Per_Week`), ou há um ponto de estagnação/retornos decrescentes?**

* **Métricas Analisadas:** Coeficiente de Correlação de Pearson, Spearman e Curva Polinomial de 2º Grau.
* **Gráfico Ideal:** Gráfico de Dispersão (Scatter Plot) com Reta de Regressão Linear e Curva de Tendência.


In [ ]:
# Cálculo das correlações e ajuste polinomial
dados_q3 = df[['AI_Usage_Hours_Per_Day', 'Time_Saved_Hours_Per_Week']].dropna()
x = dados_q3['AI_Usage_Hours_Per_Day']
y = dados_q3['Time_Saved_Hours_Per_Week']

corr_pearson = round(float(x.corr(y, method='pearson')), 3)
corr_spearman = round(float(x.rank().corr(y.rank())), 3)

poly_coefs = np.polyfit(x, y, deg=2)
poly_fn = np.poly1d(poly_coefs)
x_curve = np.linspace(x.min(), x.max(), 100)

print(f"Correlação Linear de Pearson (r): {corr_pearson}")
print(f"Correlação Monotônica de Spearman (r): {corr_spearman}")
print(f"Equação do Polinômio grau 2: y = {poly_coefs[0]:.4f}x² + {poly_coefs[1]:.4f}x + {poly_coefs[2]:.4f}")


In [ ]:
# Visualização gráfica - Pergunta 3
fig, ax = plt.subplots(figsize=(10, 6))

sns.regplot(
    x=x, y=y,
    scatter_kws={'alpha': 0.6, 'color': '#1f77b4'},
    line_kws={'color': '#d62728', 'linewidth': 2, 'label': f'Tendência Linear (Pearson r = {corr_pearson})'},
    ax=ax
)

ax.plot(x_curve, poly_fn(x_curve), color='#2ca02c', linestyle='--', linewidth=2.5, label='Curva Polinomial (Retornos Decrescentes)')

ax.set_title('Tempo Diário com IA vs. Horas Semanais Economizadas', fontweight='bold')
ax.set_xlabel('Horas Diárias Usando IA')
ax.set_ylabel('Horas Semanais Economizadas')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


**💡 Insights e Conclusões (Pergunta 3):**
- **Forte Correlação Positiva:** O coeficiente de Pearson ($r = 0.775$) e Spearman ($r = 0.824$) indicam uma relação substancial entre tempo dedicado e horas economizadas.
- **Saturação e Retornos Decrescentes:** O coeficiente quadrático negativo do polinômio ($a pprox -0.093$) confirma a hipótese de **retornos decrescentes**: até ~4 a 6 horas diárias de uso de IA, o ganho de tempo por semana cresce com vigor; após esse limite, a curva achata, indicando que gastar o dia inteiro interagindo com IA não gera ganhos adicionais proporcionais (devido a revisões, prompts repetidos ou distração).


---
### Pergunta 4: Finalidade de Uso e Satisfação
> **Quais casos de uso (`AI_Purpose` – Ex: Coding, Writing, Research, Data Analysis) apresentam os maiores índices de satisfação (`Satisfaction_Score`) e acurácia percebida (`Accuracy_Rating`)?**

* **Métricas Analisadas:** Médias de Satisfação (1 a 10) e Acurácia (1 a 5) agrupadas pela tarefa executada.
* **Gráfico Ideal:** Mapa de Calor (Heatmap) com anotação numérica direta das notas médias.


In [ ]:
# Tabela agregada por finalidade
resumo_q4 = df.groupby('AI_Purpose')[['Satisfaction_Score', 'Accuracy_Rating']].agg(
    ['mean', 'count']
).round(2)

matriz_heatmap = df.groupby('AI_Purpose')[['Satisfaction_Score', 'Accuracy_Rating']].mean().sort_values('Satisfaction_Score', ascending=False).round(2)
matriz_heatmap.columns = ['Satisfação Média (0-10)', 'Acurácia Média (1-5)']

display(resumo_q4)


In [ ]:
# Visualização gráfica - Pergunta 4 (Heatmap)
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    matriz_heatmap, annot=True, fmt=".2f", cmap="YlGnBu",
    linewidths=0.5, cbar_kws={'label': 'Nota Média'}, ax=ax
)

ax.set_title('Satisfação e Acurácia Média por Propósito de Uso da IA', fontweight='bold')
ax.set_xlabel('Métrica Avaliada')
ax.set_ylabel('Propósito de Uso (AI Purpose)')

plt.tight_layout()
plt.show()


**💡 Insights e Conclusões (Pergunta 4):**
- **Casos de Maior Satisfação:** **Research** (6.41) e **Writing** (6.29) lideram o ranking de satisfação. São atividades baseadas em texto/linguagem natural onde os LLMs entregam alto valor percebido de síntese e redação.
- **O Paradoxo da Programação (Coding):** Apesar de ser a aplicação com maior volume de usuários ($n = 65$), **Coding** obteve o menor índice de satisfação relativo (**5.82**) e acurácia percebida moderada (**3.30**). Isso reflete a alta exigência do desenvolvimento de software, onde pequenos erros de sintaxe ou alucinações de lógica penalizam a avaliação do usuário.


---
### Pergunta 5: Investimento Financeiro em IA (Planos Pagos vs. Gratuitos)
> **Usuários que investem financeiramente em planos pagos (`Monthly_AI_Cost` > $0) conseguem obter notas significativamente maiores de produtividade e economia de tempo em comparação a usuários em planos gratuitos?**

*Nota metodológica: A descrição textual original mencionava `AI_Experience_Years`, mas a coluna correspondente no dataset oficial do Kaggle é `Monthly_AI_Cost`. Analisamos o impacto financeiro (Free vs Paid) sobre os ganhos de produtividade e tempo economizado.*

* **Métricas Analisadas:** Produtividade e Economia de Tempo estratificados por `Paid_User`.
* **Gráfico Ideal:** Boxplot Comparativo Lado a Lado.


In [ ]:
# Resumo estatístico de Usuários Pagos vs Gratuitos
resumo_q5 = df.groupby('Paid_User')[['Productivity_Score', 'Time_Saved_Hours_Per_Week', 'Monthly_AI_Cost']].agg(
    ['count', 'mean', 'median', 'std']
).round(2)

display(resumo_q5)


In [ ]:
# Visualização gráfica - Pergunta 5
df_q5 = df.dropna(subset=['Paid_User', 'Productivity_Score', 'Time_Saved_Hours_Per_Week'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Produtividade
sns.boxplot(
    data=df_q5, x='Paid_User', y='Productivity_Score',
    hue='Paid_User', legend=False, palette=['#2b9348', '#55a630'], ax=axes[0]
)
axes[0].set_title('Produtividade: Gratuitos vs. Pagos', fontweight='bold')
axes[0].set_xlabel('Modalidade de Uso')
axes[0].set_ylabel('Productivity Score (0 a 10)')

# Economia de Tempo
sns.boxplot(
    data=df_q5, x='Paid_User', y='Time_Saved_Hours_Per_Week',
    hue='Paid_User', legend=False, palette=['#2b9348', '#55a630'], ax=axes[1]
)
axes[1].set_title('Tempo Economizado Semanal: Gratuitos vs. Pagos', fontweight='bold')
axes[1].set_xlabel('Modalidade de Uso')
axes[1].set_ylabel('Horas Semanais Economizadas')

plt.tight_layout()
plt.show()


**💡 Insights e Conclusões (Pergunta 5):**
- **Paridade de Desempenho:** Usuários em planos gratuitos relatam produtividade média de **5.40** e mediana de **5.10**, enquanto assinantes de planos pagos reportam média de **5.05** e mediana de **4.90**.
- **Expectativa vs. Entrega:** Usuários pagantes investem em média ~\$21/mês e tendem a ter um critério mais rigoroso de avaliação sobre o que consideram um salto significativo de produtividade. Usuários em planos gratuitos demonstram alta satisfação com os recursos já inclusos sem custo financeiro.


---
# 🚀 ÁREA RESERVADA PARA AS PERGUNTAS 6 A 10
> **👋 Olá, amigo!**  
> As etapas de carregamento, limpeza e as Perguntas 1 a 5 já estão prontas acima.  
> Você já tem o DataFrame limpo e padronizado pronto para uso na variável **`df`**.  
>  
> Abaixo deixamos um template organizado para cada uma das perguntas de 6 a 10 com as variáveis recomendadas, sugestão de gráfico e espaço para você rodar o código e escrever suas conclusões. Bom trabalho!


---
### Pergunta 6: Acurácia Percebida por Ferramenta
> **Existe discrepância estatística na acurácia reportada (`Accuracy_Rating`) entre as diferentes ferramentas de IA (ChatGPT, Gemini, Claude, Copilot, DeepSeek, etc.)?**

* **Variáveis:** `AI_Tool` (categórica), `Accuracy_Rating` (numérica de 1 a 5).
* **Sugestão de Gráfico:** `sns.boxplot` ou `sns.barplot` com ordenação das ferramentas pela mediana/média de acurácia.


In [ ]:
# TODO: Insira o código da Pergunta 6 aqui
# Dica:
# 1. Agrupar por 'AI_Tool' e calcular média/mediana de 'Accuracy_Rating'
# 2. Plotar gráfico comparativo com seaborn

# resumo_p6 = df.groupby('AI_Tool')['Accuracy_Rating'].agg(['count', 'mean', 'median', 'std']).sort_values('mean', ascending=False)
# display(resumo_p6)

# plt.figure(figsize=(10, 6))
# sns.boxplot(data=df.dropna(subset=['AI_Tool', 'Accuracy_Rating']), y='AI_Tool', x='Accuracy_Rating', palette='Blues_r')
# plt.title('Acurácia Percebida por Ferramenta de IA')
# plt.show()


**💡 Insights e Conclusões (Pergunta 6):**
*(Escreva aqui sua análise e interpretação dos resultados da Pergunta 6)*


---
### Pergunta 7: Renda Mensal e Áreas de Atuação
> **Profissionais de áreas que dependem fortemente de IA (como Tecnologia, Dados e Desenvolvimento) apresentam rendas mensais (`Monthly_Income`) superiores àqueles que usam para suporte ou tarefas administrativas?**

* **Variáveis:** `Monthly_Income` (numérica), `Profession` ou `AI_Purpose` (filtrar preferencialmente `User_Type == 'Professional'`).
* **Sugestão de Gráfico:** `sns.barplot` ou `sns.boxplot` de `Monthly_Income` por área de atuação.


In [ ]:
# TODO: Insira o código da Pergunta 7 aqui
# Dica:
# 1. Filtrar df[df['User_Type'] == 'Professional']
# 2. Agrupar renda por AI_Purpose ou agrupar carreiras de tecnologia
# 3. Comparar as medianas salariais

# prof_df = df[df['User_Type'] == 'Professional'].dropna(subset=['Monthly_Income', 'AI_Purpose'])
# resumo_p7 = prof_df.groupby('AI_Purpose')['Monthly_Income'].agg(['count', 'mean', 'median']).sort_values('median', ascending=False)
# display(resumo_p7)


**💡 Insights e Conclusões (Pergunta 7):**
*(Escreva aqui sua análise e interpretação dos resultados da Pergunta 7)*


---
### Pergunta 8: Nível Educacional e Adoção de IA para Pesquisa
> **O uso de IA para *Research* e *Studying* é mais prevalente e avaliado de forma diferente entre estudantes de graduação versus pós-graduação/mestrado/PhD?**

* **Variáveis:** `Education_Level` (categórica), `AI_Purpose` (foco em 'Research' e 'Studying'), `Satisfaction_Score` ou proporção de uso.
* **Sugestão de Gráfico:** Gráfico de Barras Empilhadas ou Barras Agrupadas (`sns.countplot` ou `pd.crosstab`).


In [ ]:
# TODO: Insira o código da Pergunta 8 aqui
# Dica:
# 1. Analisar a distribuição de AI_Purpose entre os diferentes Education_Level
# 2. Calcular a proporção de estudantes que utilizam para 'Research' e 'Studying'

# cross_p8 = pd.crosstab(df['Education_Level'], df['AI_Purpose'], normalize='index') * 100
# display(cross_p8.round(1))


**💡 Insights e Conclusões (Pergunta 8):**
*(Escreva aqui sua análise e interpretação dos resultados da Pergunta 8)*


---
### Pergunta 9: Impacto no Atendimento e Rotinas Corporativas
> **No setor de *Customer Support* e *Office Work*, qual é a média de tempo semanal economizado e qual o impacto percebido na produtividade?**

* **Variáveis:** `AI_Purpose` (filtrando `['Customer Support', 'Office Work']`), `Time_Saved_Hours_Per_Week`, `Productivity_Score`.
* **Sugestão de Gráfico:** `sns.barplot` com intervalos de confiança ou Boxplot comparativo em relação à média geral.


In [ ]:
# TODO: Insira o código da Pergunta 9 aqui
# Dica:
# 1. Criar filtro para df['AI_Purpose'].isin(['Customer Support', 'Office Work'])
# 2. Comparar as métricas de tempo economizado e produtividade desse grupo vs restante da base

# suporte_office = df[df['AI_Purpose'].isin(['Customer Support', 'Office Work'])]
# display(suporte_office[['Time_Saved_Hours_Per_Week', 'Productivity_Score']].describe().round(2))


**💡 Insights e Conclusões (Pergunta 9):**
*(Escreva aqui sua análise e interpretação dos resultados da Pergunta 9)*


---
### Pergunta 10: Faixa Etária e Padrão de Adoção
> **Como a idade (`Age`) influencia a escolha da ferramenta de IA e a percepção geral de satisfação com as soluções adotadas?**

* **Variáveis:** `Age` (numérica contínua ou discretizada em faixas: `< 20`, `20-29`, `30-39`, `40+`), `AI_Tool`, `Satisfaction_Score`.
* **Sugestão de Gráfico:** Dispersão `sns.scatterplot` com linha de tendência ou `sns.boxplot` por faixas etárias.


In [ ]:
# TODO: Insira o código da Pergunta 10 aqui
# Dica:
# 1. Criar faixas etárias usando pd.cut(df['Age'], bins=[0, 20, 30, 40, 100], labels=['<20', '20-29', '30-39', '40+'])
# 2. Analisar ferramenta predominante e média de satisfação por faixa etária

# df['Faixa_Etaria'] = pd.cut(df['Age'], bins=[0, 20, 30, 40, 100], labels=['<20', '20-29', '30-39', '40+'])
# resumo_p10 = df.groupby('Faixa_Etaria')['Satisfaction_Score'].agg(['count', 'mean', 'median'])
# display(resumo_p10)


**💡 Insights e Conclusões (Pergunta 10):**
*(Escreva aqui sua análise e interpretação dos resultados da Pergunta 10)*
